<a href="https://colab.research.google.com/github/waleed-87/NLP/blob/main/txt_classfi_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
temp=pd.read_csv('/content/sample_data/IMDB Dataset.csv')

In [ ]:
df=temp.iloc[:10000]

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_1897/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [ ]:
import re
def removed_tags(raw_txt):
  cleaned_txt=re.sub('<.*?>','',raw_txt)
  return cleaned_txt

In [ ]:
df['review']=df['review'].apply(removed_tags)

/tmp/ipykernel_1897/2688771884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(removed_tags)


In [ ]:
df['review']=df['review'].apply(lambda x: x.lower())

/tmp/ipykernel_1897/3883677782.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(lambda x: x.lower())


In [ ]:
!pip install nltk

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
sw_list=stopwords.words('english')
df.loc[:, 'review']=df['review'].apply(lambda x :[item for item in x.split() if item not in sw_list]).apply(lambda x: " ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
!pip install gensim


In [ ]:

from nltk import sent_tokenize
from gensim.utils import simple_preprocess
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

story=[]
for doc in df['review']:
  raw_sent=sent_tokenize(doc)
  for sent in raw_sent:
    story.append(simple_preprocess(sent))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:

import gensim

model=gensim.models.Word2Vec(vector_size=100, window=10,min_count=2)

In [ ]:
model.build_vocab(story)

In [ ]:
model.train(story,total_examples=model.corpus_count,epochs=model.epochs)

(5849437, 6186875)

In [ ]:
len(model.wv.index_to_key)

31845

In [ ]:
def document_vector(doc):
  doc=[word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc],axis=0)

In [ ]:

document_vector(df['review'].values[0])

array([-9.7939663e-02,  3.5779801e-01,  1.2957835e-01, -8.5633397e-02,
       -3.0132364e-03, -4.9078435e-01,  1.2853073e-01,  8.6219895e-01,
       -3.0152610e-01, -1.0034567e-02, -1.9881988e-01, -5.5591565e-01,
        2.9668331e-02,  1.3130710e-01,  1.1134587e-01, -3.1617793e-01,
        1.6234614e-01, -5.3428036e-01,  1.2810193e-01, -7.7459455e-01,
        1.5138936e-01,  5.5696912e-02,  1.7835325e-01, -2.0424786e-01,
       -2.5516990e-01, -1.4263101e-01, -1.9255638e-01, -2.1874423e-01,
       -3.3120564e-01,  1.1913478e-05,  4.3190518e-01,  3.7673081e-03,
        2.9868651e-02, -2.7777535e-01, -2.0074835e-01,  3.3095059e-01,
        1.4618853e-01, -3.6433756e-01, -1.9485575e-01, -6.8488330e-01,
        1.2571263e-01, -2.5423288e-01, -1.1438589e-01, -2.0578235e-02,
        4.1220102e-01, -1.3551454e-01, -4.2400789e-01, -1.0869063e-01,
        1.9357030e-01,  2.4657136e-01,  1.4442904e-01, -2.9770571e-01,
       -2.9691353e-01, -1.7276621e-01, -3.5097101e-01,  8.4101304e-02,
      

In [ ]:

from tqdm import tqdm

In [ ]:
X=[]
for doc in tqdm(df['review'].values):
  X.append(document_vector(doc))

100%|██████████| 9983/9983 [06:38<00:00, 25.06it/s]


In [ ]:
X=np.array(X)

In [ ]:
X[0]

array([-9.7939663e-02,  3.5779801e-01,  1.2957835e-01, -8.5633397e-02,
       -3.0132364e-03, -4.9078435e-01,  1.2853073e-01,  8.6219895e-01,
       -3.0152610e-01, -1.0034567e-02, -1.9881988e-01, -5.5591565e-01,
        2.9668331e-02,  1.3130710e-01,  1.1134587e-01, -3.1617793e-01,
        1.6234614e-01, -5.3428036e-01,  1.2810193e-01, -7.7459455e-01,
        1.5138936e-01,  5.5696912e-02,  1.7835325e-01, -2.0424786e-01,
       -2.5516990e-01, -1.4263101e-01, -1.9255638e-01, -2.1874423e-01,
       -3.3120564e-01,  1.1913478e-05,  4.3190518e-01,  3.7673081e-03,
        2.9868651e-02, -2.7777535e-01, -2.0074835e-01,  3.3095059e-01,
        1.4618853e-01, -3.6433756e-01, -1.9485575e-01, -6.8488330e-01,
        1.2571263e-01, -2.5423288e-01, -1.1438589e-01, -2.0578235e-02,
        4.1220102e-01, -1.3551454e-01, -4.2400789e-01, -1.0869063e-01,
        1.9357030e-01,  2.4657136e-01,  1.4442904e-01, -2.9770571e-01,
       -2.9691353e-01, -1.7276621e-01, -3.5097101e-01,  8.4101304e-02,
      

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
y=encoder.fit_transform(df['sentiment'])

In [ ]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
rf=RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.771657486229344